# Telco Customer Churn Prediction
### Identifying at-risk customers with Logistic Regression, Decision Tree and Random Forest

## Introduction

Customer churn (customers leaving the service) directly reduces revenue, and retaining an existing customer is typically cheaper than acquiring a new one. This project analyzes the IBM Telco Customer Churn dataset to understand **which customer characteristics are associated with churn** and to build a model that **flags customers who are likely to churn**, so that retention efforts can be targeted.

**Goals**
1. Explore the data and identify patterns associated with churn.
2. Build and compare classification models, with a focus on correctly identifying the churn class.
3. Handle class imbalance and choose a decision threshold that fits the retention use case.
4. Translate the findings into business recommendations.

## Dataset

The dataset contains 7,043 customers and 21 columns (file: `WA_Fn-UseC_-Telco-Customer-Churn.csv`, the IBM sample dataset, available on Kaggle).

| Group | Columns |
|---|---|
| Customer | `gender`, `SeniorCitizen`, `Partner`, `Dependents` |
| Account | `tenure`, `Contract`, `PaperlessBilling`, `PaymentMethod`, `MonthlyCharges`, `TotalCharges` |
| Services | `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` |
| Identifier | `customerID` (dropped, no predictive meaning) |
| **Target** | `Churn` (Yes = the customer left within the last month) |

## Workflow

Data inspection and cleaning, exploratory data analysis (EDA), task definition, train/validation/test split, modeling, evaluation (including threshold tuning and interpretation), and conclusion.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, precision_recall_curve,
                             ConfusionMatrixDisplay, RocCurveDisplay)

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# 1. Data Loading, Inspection and Cleaning

### 1.1. Inspect Data

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe().T

**Findings (Data Inspection)**
- The dataset has 7,043 customers and 21 columns, with no missing values and no duplicated rows.
- `TotalCharges` is stored as text, so it is excluded from `.describe()` and must be converted to numeric during cleaning.
- `tenure` ranges from 0 to 72 months (median 29), mixing new and long-term customers.
- `MonthlyCharges` ranges from 18.25 to 118.75 (median 70.35, mean 64.76).
- `SeniorCitizen` is binary, with about 16% senior customers.

### 1.2. Data Cleaning

In [ ]:
# TotalCharges is text because some values are blank. Check which customers they belong to.
blank = df['TotalCharges'].str.strip() == ''
print('Blank TotalCharges:', blank.sum())
print('Tenure of these customers:', df.loc[blank, 'tenure'].unique())

In [ ]:
# Blank TotalCharges belong to customers with tenure 0 (no bill yet), so 0 is a reasonable value
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip().replace('', '0'))
df['TotalCharges'].dtype

In [ ]:
# Check value counts of categorical columns to spot inconsistent wording
for col in df.select_dtypes(exclude='number').columns.drop(['customerID', 'Churn']):
    print(f'___{col}___')
    print(df[col].value_counts())
    print()

In [ ]:
# 'No phone service' and 'No internet service' only repeat information already in
# PhoneService and InternetService, so they are merged into 'No'
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')
internet_addons = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                   'TechSupport', 'StreamingTV', 'StreamingMovies']
for c in internet_addons:
    df[c] = df[c].replace('No internet service', 'No')

print(df['MultipleLines'].value_counts(), '\n')
for c in internet_addons:
    print(df[c].value_counts(), '\n')

In [ ]:
# customerID is a unique identifier with no predictive meaning
df = df.drop('customerID', axis=1)

# Encode the target: 1 = churned, 0 = stayed
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print(df['Churn'].value_counts())
print(df.shape)

**Findings (Data Cleaning)**
- 11 blank `TotalCharges` values belong to customers with tenure 0, so they were set to 0 and the column converted to numeric.
- `No phone service` and `No internet service` were merged into `No` (the information is kept in `PhoneService` and `InternetService`).
- `customerID` was dropped and `Churn` encoded as 1 (churned) and 0 (stayed).

# 2. Exploratory Data Analysis (EDA)

In [ ]:
num_col = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_col = [c for c in df.columns if c not in num_col + ['Churn']]   # includes SeniorCitizen (binary)

### 2.1. Target Distribution

In [ ]:
ax = sns.countplot(x='Churn', data=df)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Stayed (0)', 'Churned (1)'])
plt.title('Distribution of Churn')
plt.show()

print(df['Churn'].value_counts().sort_index())
print(round(df['Churn'].value_counts(normalize=True).sort_index() * 100, 2))

**Findings**
The target is imbalanced: about 73.5% of customers stayed (Churn = 0) and 26.5% left (Churn = 1). A model that always predicts `stayed` would already reach about 73.5% accuracy, so accuracy alone is misleading. Precision, recall and F1 for the churn class are used instead.

### 2.2. Feature Distributions

In [ ]:
df[num_col].hist(figsize=(12, 4), bins=20, color='purple', layout=(1, 3))
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()
for ax, col in zip(axes, cat_col):
    df[col].value_counts().plot(kind='bar', ax=ax, color='skyblue')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)
for ax in axes[len(cat_col):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

### 2.3. Churn vs Numerical Features

In [ ]:
units = {'tenure': 'Months', 'MonthlyCharges': 'USD per month', 'TotalCharges': 'USD'}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, num_col):
    ax.hist(df[df['Churn'] == 1][col], color='red', label='Churned', alpha=0.7, density=True, bins=20)
    ax.hist(df[df['Churn'] == 0][col], color='blue', label='Stayed', alpha=0.7, density=True, bins=20)
    ax.set_title(col)
    ax.set_ylabel('Density')
    ax.set_xlabel(f'{col} ({units[col]})')
    ax.legend()
plt.tight_layout()
plt.show()

**Findings**
- **Tenure:** Churned customers are concentrated in the first months, while long-term customers (about 65-72 months) are mostly customers who stayed.
- **MonthlyCharges:** Churned customers are over-represented at high monthly charges (roughly 70-110 USD), while customers who stayed are concentrated at the lowest charges (about 20 USD). The two groups overlap between about 30 and 70 USD.
- **TotalCharges:** Churned customers are concentrated at low total charges, which is consistent with their short tenure.
- **Overall:** Churn is associated with short tenure and higher monthly charges.

In [ ]:
def churn_rate_by_bins(col, bins, labels, xlabel):
    groups = pd.cut(df[col], bins=bins, labels=labels, include_lowest=True)
    rate = df.groupby(groups, observed=False)['Churn'].mean()

    ax = rate.plot(kind='bar', color='grey', alpha=0.7)
    ax.bar_label(ax.containers[0], fmt='%.2f')
    plt.axhline(df['Churn'].mean(), color='black', linestyle='--', label='Overall churn rate')
    plt.xlabel(xlabel)
    plt.ylabel('Churn rate')
    plt.title(f'Churn Rate by {col} Group')
    plt.xticks(rotation=0)
    plt.legend()
    plt.show()

In [ ]:
# Do customers with a shorter tenure have a higher churn rate?
churn_rate_by_bins('tenure', [-1, 6, 12, 24, 48, 72],
                   ['0-6', '7-12', '13-24', '25-48', '49-72'], 'Tenure (months)')

**Findings:** Churn rate decreases steadily with tenure. Customers in their first 6 months churn at about 53%, roughly twice the overall rate (about 26.5%), while customers with 49-72 months churn at only about 10%. Customers in their first 12 months are the highest-risk group (53% and 36%), and the churn rate falls below the overall average after about 24 months. This is an association, not evidence that short tenure causes churn.

In [ ]:
# Does the churn rate change with monthly charges?
churn_rate_by_bins('MonthlyCharges', [0, 30, 50, 70, 90, 120],
                   ['0-30', '30-50', '50-70', '70-90', '90-120'], 'Monthly charges (USD per month)')

**Findings:** Churn rate varies by monthly charge group. The highest churn is in the 70-90 USD group (38%) and the lowest in the 0-30 USD group (10%). The 30-50 (31%) and 90-120 (33%) groups are also above the overall average, while 50-70 is below it (21%). Because churn does not increase consistently with charges, the relationship is not linear.

In [ ]:
# Do customers with higher total charges have a higher or lower churn rate?
churn_rate_by_bins('TotalCharges', [-1, 1000, 2000, 4000, 6000, 9000],
                   ['0-1000', '1001-2000', '2001-4000', '4001-6000', '6001-9000'], 'Total charges (USD)')

**Findings:** Customers with the lowest total charges (0-1000 USD) have the highest churn rate at 37%, while those with 6001-9000 USD have the lowest at 13%. Since total charges accumulate over time, this is largely a reflection of tenure (see the next plot).

In [ ]:
# Is the relationship between TotalCharges and Churn influenced by tenure?
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='tenure', y='TotalCharges', hue='Churn', alpha=0.4)
plt.title('Relationship between Tenure and Total Charges')
plt.xlabel('Tenure (months)')
plt.ylabel('Total charges (USD)')
plt.tight_layout()
plt.show()

**Findings:** Total charges grow with tenure (roughly tenure x monthly charge), so the two features carry overlapping information.

### 2.4. Churn vs Categorical Features

In [ ]:
rows = []
for col in cat_col:
    rates = df.groupby(col)['Churn'].mean()
    rows.append({'Feature': col, 'Min churn': rates.min(), 'Max churn': rates.max(),
                 'Gap': rates.max() - rates.min()})
pd.DataFrame(rows).sort_values('Gap', ascending=False).round(3).reset_index(drop=True)

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 14))
axes = axes.flatten()
for ax, col in zip(axes, cat_col):
    df.groupby(col)['Churn'].mean().plot(kind='bar', color='brown', alpha=0.7, ax=ax)
    ax.set_title(col)
    ax.set_ylim(0, 0.5)
    ax.set_ylabel('Churn rate')
    ax.tick_params(axis='x', rotation=30)
for ax in axes[len(cat_col):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

**Findings**
- **Contract** is the strongest separator: month-to-month customers churn at about 43%, compared with about 11% for one-year and about 3% for two-year contracts.
- **InternetService:** fiber optic customers churn at about 42%, compared with about 19% for DSL and about 7% for customers without internet.
- **PaymentMethod:** customers paying by electronic check churn at about 45%, roughly three times the rate of other methods (15-19%).
- **Add-on services:** customers without `OnlineSecurity` or `TechSupport` churn at about 31%, versus about 15% for those who have them. `OnlineBackup` and `DeviceProtection` show smaller gaps (6-8 percentage points).
- **Customer profile:** senior citizens churn more (about 42% vs 24%), and customers without a partner (about 33% vs 20%) or dependents (about 31% vs 16%) also churn more. `PaperlessBilling` is associated with higher churn (about 34% vs 16%).
- **Small or no difference:** `gender`, `PhoneService`, `MultipleLines`, `StreamingTV` and `StreamingMovies`.

### 2.5. Correlation Analysis

In [ ]:
corr = df.corr(numeric_only=True, method='pearson')
plt.figure(figsize=(8, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

In [ ]:
# Pearson vs Spearman correlation with Churn (do non-linear monotonic effects change the picture?)
cmp = pd.DataFrame({
    'pearson': df.corr(numeric_only=True, method='pearson')['Churn'],
    'spearman': df.corr(numeric_only=True, method='spearman')['Churn'],
}).drop('Churn')
cmp['gap'] = (cmp['spearman'] - cmp['pearson']).abs()
print(cmp.round(3).sort_values('gap', ascending=False))

**Findings:** `tenure` has the strongest (negative) correlation with churn (r = -0.35), followed by `TotalCharges` (-0.20, mostly through tenure), `MonthlyCharges` (+0.19) and `SeniorCitizen` (+0.15). Pearson and Spearman values differ by at most 0.03, so the monotonic relationships are close to linear. All correlations with churn are weak to moderate, so no single numerical feature explains churn on its own.

In [ ]:
def cramers_v(x, y):
    table = pd.crosstab(x, y)
    if min(table.shape) < 2:
        return np.nan
    chi2 = chi2_contingency(table, correction=False)[0]
    n = table.sum().sum()
    r, k = table.shape
    return np.sqrt(chi2 / (n * min(k - 1, r - 1)))

In [ ]:
cramers_cols = df.select_dtypes(exclude='number').columns.tolist() + ['SeniorCitizen', 'Churn']

cramers_matrix = pd.DataFrame(np.nan, index=cramers_cols, columns=cramers_cols, dtype=float)
for a in cramers_cols:
    for b in cramers_cols:
        cramers_matrix.loc[a, b] = cramers_v(df[a], df[b])

plt.figure(figsize=(9, 7))
sns.heatmap(cramers_matrix, annot=True, fmt='.2f', cmap='Reds', vmin=0, vmax=1)
plt.title("Cramer's V between categorical columns")
plt.tight_layout()
plt.show()

**Findings:** Categorical features show only moderate associations with each other, mostly between related services (for example `StreamingTV` and `StreamingMovies` at about 0.53), so no pair is close to redundant. The categorical features most associated with churn are `Contract` (about 0.41), `InternetService` (about 0.32) and `PaymentMethod` (about 0.30), which is consistent with the churn rates in section 2.4.

# 3. Task Definition

This is a **binary classification** problem: predict whether a customer churns (1) or stays (0). The positive class is **churn**. Because only about 26.5% of customers churn, models are judged mainly by **precision, recall and F1 for the churn class** and by **ROC-AUC**, not by accuracy.

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

numeric_features = X.select_dtypes(include='number').columns.tolist()
categorical_features = X.select_dtypes(exclude='number').columns.tolist()

assert 'Churn' not in X.columns   # the target must not be among the features
print('Numeric:', numeric_features)
print('Categorical:', categorical_features)

# 4. Train / Validation / Test Split

In [ ]:
# 60% train, 20% validation, 20% test (stratified to keep the churn rate in every part)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=RANDOM_STATE, stratify=y)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp)

for name, part in [('Train', y_train), ('Validation', y_valid), ('Test', y_test)]:
    print(f'{name:<11} rows: {len(part):>5}   churn rate: {part.mean():.3f}')

The validation set is used to compare models and to choose the decision threshold. The test set is used **once**, at the end.

### 4.1. Preprocessing

In [ ]:
# Logistic Regression: scale numeric features and one-hot encode categorical features
preprocessor_scaled = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
])

# Tree-based models: no scaling needed, only one-hot encoding
preprocessor_tree = ColumnTransformer([
    ('num', 'passthrough', numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
])

Both preprocessors are placed inside a `Pipeline`, so they are fitted on the training data only (no data leakage).

# 5. Modeling

### 5.1. Baseline: Dummy Classifier

The baseline always predicts the majority class (stayed). Any useful model must beat it on the churn class.

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)

### 5.2. Models

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),
    'Logistic Regression (balanced)': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))]),
    'Decision Tree': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', DecisionTreeClassifier(random_state=RANDOM_STATE))]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))]),
}

`class_weight='balanced'` gives the minority (churn) class more weight during training, as one way to handle the class imbalance.

### 5.3. Cross-Validation on the Training Set

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['precision', 'recall', 'f1', 'roc_auc']     # computed for the churn class (label 1)

rows = []
for name, model in models.items():
    cv = cross_validate(model, X_train, y_train, cv=skf, scoring=scoring)
    row = {'Model': name}
    for s in scoring:
        scores = cv[f'test_{s}']
        row[s] = f'{scores.mean():.3f} (+/- {scores.std():.3f})'
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

Stratified 5-fold cross-validation on the training data gives a more stable comparison than a single split. The mean and standard deviation across folds are shown for the churn class.

# 6. Evaluation

### 6.1. Model Comparison on the Validation Set

All metrics below are for the **churn class** (label 1).

In [ ]:
def evaluate(model, X_eval, y_eval, threshold=0.5):
    prob = model.predict_proba(X_eval)[:, 1]            # probability of churn
    pred = (prob >= threshold).astype(int)
    return {
        'Accuracy': accuracy_score(y_eval, pred),
        'Precision (Churn)': precision_score(y_eval, pred, zero_division=0),
        'Recall (Churn)': recall_score(y_eval, pred, zero_division=0),
        'F1 (Churn)': f1_score(y_eval, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_eval, prob),
    }

In [ ]:
results = [{'Model': 'Baseline (Dummy)', **evaluate(baseline, X_valid, y_valid)}]
for name, model in models.items():
    model.fit(X_train, y_train)                         # fit on the training set only
    results.append({'Model': name, **evaluate(model, X_valid, y_valid)})

results_df = pd.DataFrame(results).sort_values('F1 (Churn)', ascending=False)
print(results_df.round(3).to_string(index=False))

**Findings (validation set)**
- Accuracy is misleading: the baseline reaches about 73.5% accuracy by predicting that nobody churns, but its recall and F1 for churn are 0.
- Among Logistic Regression, Random Forest and Decision Tree at default settings, Logistic Regression has the best churn F1 (about 0.64) and ROC-AUC (about 0.86). The Decision Tree is the weakest (F1 about 0.51, ROC-AUC about 0.67).
- The balanced variant shows how class weighting trades precision for recall.

### 6.2. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (name, model) in zip(axes.flatten(), models.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_valid, y_valid,
                                          display_labels=['Stayed', 'Churn'], ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

In each matrix, the bottom-left cell counts churned customers predicted as staying (missed churners), and the top-right cell counts customers who stayed but were flagged as churn (false alarms).

### 6.3. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, model in models.items():
    RocCurveDisplay.from_estimator(model, X_valid, y_valid, name=name, ax=ax)
ax.plot([0, 1], [0, 1], 'k--', label='Random guess')
ax.set_title('ROC Curves (validation set)')
ax.legend()
plt.show()

### 6.4. Overfitting Check

In [ ]:
rows = []
for name, model in models.items():
    tr = evaluate(model, X_train, y_train)
    va = evaluate(model, X_valid, y_valid)
    rows.append({'Model': name,
                 'F1 train': tr['F1 (Churn)'], 'F1 valid': va['F1 (Churn)'],
                 'F1 gap': tr['F1 (Churn)'] - va['F1 (Churn)'],
                 'AUC train': tr['ROC-AUC'], 'AUC valid': va['ROC-AUC'],
                 'AUC gap': tr['ROC-AUC'] - va['ROC-AUC']})
print(pd.DataFrame(rows).round(3).to_string(index=False))

A large gap between training and validation scores means the model memorizes the training data. The fully grown Decision Tree and Random Forest fit the training data almost perfectly, so their gaps are large, whereas Logistic Regression stays close to its validation score.

### 6.5. Model Selection and Threshold Tuning

The model with the highest validation F1 for churn is selected. By default a customer is flagged as churn when the predicted probability is at least 0.5. Because missing a churner is usually more costly than a false alarm, the threshold is tuned on the **validation set** to maximize churn F1.

In [ ]:
candidates = results_df[results_df['Model'] != 'Baseline (Dummy)']
best_name = candidates.iloc[0]['Model']
best_model = models[best_name]
print('Selected model:', best_name)

prob_valid = best_model.predict_proba(X_valid)[:, 1]
prec, rec, thr = precision_recall_curve(y_valid, prob_valid)
f1_curve = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_thr = round(float(thr[f1_curve.argmax()]), 2)
print('Threshold that maximizes validation F1:', best_thr)

rows = []
for t in sorted({0.3, 0.4, 0.5, best_thr}):
    m = evaluate(best_model, X_valid, y_valid, threshold=t)
    rows.append({'Threshold': t, 'Precision (Churn)': m['Precision (Churn)'],
                 'Recall (Churn)': m['Recall (Churn)'], 'F1 (Churn)': m['F1 (Churn)']})
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(thr, prec[:-1], label='Precision')
plt.plot(thr, rec[:-1], label='Recall')
plt.plot(thr, f1_curve, label='F1')
plt.axvline(best_thr, color='black', linestyle='--', label=f'Selected threshold = {best_thr}')
plt.xlabel('Decision threshold')
plt.ylabel('Score (churn class)')
plt.title('Precision, Recall and F1 vs Threshold (validation set)')
plt.legend()
plt.show()

Lowering the threshold increases recall (more churners are found) at the cost of precision (more false alarms). The final threshold should ultimately reflect the cost of a retention offer versus the revenue lost from a churned customer.

### 6.6. Final Evaluation on the Test Set

In [ ]:
# The test set is only reported here, it was not used to choose the model or the threshold
final = {
    'Logistic Regression @ 0.5': evaluate(models['Logistic Regression'], X_test, y_test, 0.5),
    f'{best_name} @ 0.5': evaluate(best_model, X_test, y_test, 0.5),
    f'{best_name} @ {best_thr} (selected)': evaluate(best_model, X_test, y_test, best_thr),
}
print(pd.DataFrame(final).T.round(3).to_string())

pred_test = (best_model.predict_proba(X_test)[:, 1] >= best_thr).astype(int)
print(f'\nClassification report ({best_name}, tuned threshold {best_thr}, test set)')
print(classification_report(y_test, pred_test, target_names=['Stayed', 'Churn'], zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, pred_test, display_labels=['Stayed', 'Churn'])
plt.title(f'Confusion Matrix - {best_name} (test set)')
plt.show()

### 6.7. Model Interpretation

In [ ]:
lr = models['Logistic Regression']
feature_names = [n.split('__', 1)[1] for n in lr.named_steps['preprocessor'].get_feature_names_out()]
coefs = pd.Series(lr.named_steps['classifier'].coef_[0], index=feature_names).sort_values()

top = pd.concat([coefs.head(10), coefs.tail(10)])
top.plot(kind='barh', figsize=(8, 7), color=['tab:blue' if v < 0 else 'tab:red' for v in top])
plt.title('Logistic Regression coefficients (red = higher churn risk, blue = lower)')
plt.xlabel('Coefficient (numeric features are standardized)')
plt.tight_layout()
plt.show()

print('Top 5 features increasing churn risk:')
print(coefs.tail(5)[::-1].round(2))
print('\nTop 5 features decreasing churn risk:')
print(coefs.head(5).round(2))

Coefficients show the direction of association after accounting for the other features. They should be read with care: several features overlap (for example `tenure` and `TotalCharges`, or `InternetService` and the add-on services), and the results describe associations, not causes.

# 7. Conclusion

### 7.1. Exploratory Data Analysis
- **Tenure:** churn is highest among new customers (about 53% in the first 6 months, versus about 10% for customers with 49-72 months).
- **Contract type:** month-to-month customers churn at about 43%, compared with about 11% (one-year) and about 3% (two-year).
- **Internet service and payment method:** fiber optic customers (about 42%) and electronic check payers (about 45%) churn the most.
- **Add-on services:** customers without online security or tech support churn at about 31%, versus about 15% for those with them.
- **Feature relationships:** categorical features show only moderate associations with each other (up to about 0.5), and the ones most associated with churn are `Contract`, `InternetService` and `PaymentMethod`.

### 7.2. Modeling and Evaluation
- The target is imbalanced (about 26.5% churn), so models were judged on churn precision, recall, F1 and ROC-AUC. A baseline that never predicts churn reaches about 73.5% accuracy but finds no churners.
- Logistic Regression, Decision Tree, Random Forest and a class-weighted Logistic Regression were compared with stratified 5-fold cross-validation and a validation set. Among the three default models, Logistic Regression performed best (validation churn recall about 0.59, precision about 0.70, F1 about 0.64, ROC-AUC about 0.86), while the fully grown Decision Tree and Random Forest overfit the training data.
- At the default 0.5 threshold, plain Logistic Regression finds only about half of the churners on the test set (churn recall about 0.53, precision about 0.62, F1 about 0.57). Class weighting and threshold tuning (sections 5.2 and 6.5) address this by trading some precision for higher recall, and the test metrics of the selected model and threshold are reported in section 6.6.

### 7.3. Business Insights and Recommendations
- **Identify at-risk customers:** use the predicted churn probabilities to rank customers and prioritize retention efforts for the highest-risk ones.
- **Focus on the first year:** new customers are the highest-risk group, so onboarding and early check-ins are likely to have the largest effect.
- **Review contract strategy:** explore incentives that move month-to-month customers to longer contracts.
- **Investigate payment and service issues:** examine why electronic check payers and fiber optic customers churn more, and whether bundling security or tech support helps.
- **Run retention experiments:** test targeted offers on high-risk customers and measure their effect.

These recommendations are based on associations, not causes, and should be validated with further analysis or A/B tests.

### 7.4. Limitations and Next Steps
- Results come from a single dataset and a single train/validation/test split.
- The 11 customers with tenure 0 have `TotalCharges` set to 0, and the data has no time dimension (no information on when or why customers left).
- Hyperparameters were not tuned, and predicted probabilities were not calibrated.
- Next steps: tune hyperparameters, try gradient boosting, calibrate probabilities, and choose the threshold from an explicit cost-benefit analysis (cost of a retention offer versus revenue lost per churned customer).

### 7.5. Overall Conclusion
This project shows an end-to-end churn workflow: careful data cleaning, EDA that identifies the main churn drivers, a model comparison that accounts for class imbalance, an honest overfitting check, threshold tuning for the retention use case, and recommendations that follow from the evidence.